# CBDTP Relevance + Stance — Anchored-Thread LLM Cascade (vLLM)
### Milestone 2 — vLLM throughput variant

Identical pipeline to `CBDTP_Relevance_Filtering_with_Choice.ipynb`, but the LLM stage runs on
**vLLM** (continuous batching + PagedAttention) instead of HuggingFace `generate()`. Expect a large
throughput gain on the same `Qwen/Qwen2.5-7B-Instruct-AWQ` weights.

**Reuses** the existing anchoring/assembly checkpoints in `relevance_filtering-7B-AWQ/` (model-agnostic,
already computed) and **writes all LLM outputs under a `-vllm` suffix**, so it is fully isolated from the
HuggingFace run (which uses `*-shard*` names) — the two can coexist in the same checkpoint dir.

**Platform:** vLLM is Linux-only — run under **WSL2** or **native Linux**. See the project plan doc
`VLLM_PLAN.md` for environment setup. This notebook will not import on native Windows.

## 0 · Run mode — pick ONE (config cell below)

Three ways to run, set by knobs in §1:

1. **Single GPU (simple, recommended baseline):** `WORKER_MODE=False`, `RUN_ON_BOTH_GPUS=False`. Runs on
   `SINGLE_GPU_INDEX` (default `"1"`), leaving the other card free to game. One 4090 + vLLM is already fast.
2. **Both GPUs, tensor-parallel:** `RUN_ON_BOTH_GPUS=True`. ⚠️ On these **non-NVLink** 4090s, TP syncs over PCIe
   and is often **slower than a single GPU and can stall** (`shm_broadcast` 60 s timeouts, `sm 100%`/`mem 0%`
   spin). Generally avoid — use mode 3 for both cards.
3. **Both GPUs, dynamic data-parallel (best for using both cards):** `WORKER_MODE=True`. Run **one full model per
   GPU** as independent workers pulling from a **shared claim queue** (§9b) — no cross-GPU comm, no static split,
   no idle when one finishes early. **Duplicate this notebook to two files**, set `WORKER_MODE=True` in both and
   `WORKER_GPU="0"` / `"1"`, then Run All on each. Stop one with **Interrupt** (releases its claim; engine stays
   loaded → re-run §9b to resume). Do the final merge from a copy with `WORKER_MODE=False`.

`GPU_MEM_UTIL` (default `0.90`; `0.82` for a worker on display-GPU 0), `MAX_NUM_SEQS` (default `128`) and
`MADS_DATA_DIR` are env-overridable.

## 1 · Configuration

In [1]:
%%time
import os, re, json, time, hashlib, warnings, math
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import sys
try: sys.stdout.reconfigure(encoding="utf-8")   # robust prints regardless of console codepage
except Exception: pass

# --- GPU selection (Linux/vLLM): set CUDA_VISIBLE_DEVICES BEFORE importing vllm/torch. This is the
#     standard vLLM way (the HF/Windows notebook avoided it because it could hang the Windows driver;
#     on Linux it is correct and expected).
# ══ GPU MODE — edit these lines ═══════════════════════════════════════════════════════════════════════
RUN_ON_BOTH_GPUS = False    # True = BOTH 4090s in ONE engine (tensor-parallel). NB: on non-NVLink 4090s this is
                           # often SLOWER and can stall on PCIe — prefer single GPU, or WORKER_MODE for both cards.
# -- Dynamic data-parallel WORKER (the good way to use BOTH cards): one full model per GPU, each pulling from a
#    SHARED claim queue (no static split, no idle when one finishes early). To use: DUPLICATE this notebook to two
#    files, set WORKER_MODE=True in both, WORKER_GPU "0" in one and "1" in the other, and Run All on each. Stop one
#    with Interrupt (it releases its claim; the engine stays loaded -> re-run the §9b cell to resume instantly).
WORKER_MODE = False
WORKER_GPU  = "1"          # "0" or "1" — MUST be set differently in each worker copy
SINGLE_GPU_INDEX = WORKER_GPU   # which card for single-GPU (non-worker) runs; follows WORKER_GPU so all knobs agree ("1" keeps GPU 0 free for BG3)
# ══════════════════════════════════════════════════════════════════════════════════════════════════════
if WORKER_MODE:
    VLLM_GPUS, TP_SIZE = WORKER_GPU, 1
elif RUN_ON_BOTH_GPUS:
    VLLM_GPUS, TP_SIZE = "0,1", 2
else:
    VLLM_GPUS, TP_SIZE = SINGLE_GPU_INDEX, 1
os.environ["CUDA_VISIBLE_DEVICES"] = VLLM_GPUS    # MUST be set before torch/vllm init -> done here, above `import torch`
# FlashInfer JIT-compiles its sampler with nvcc+ninja; on WSL's older apt CUDA toolkit that build fails.
# Use vLLM's native PyTorch sampler instead (negligible perf cost). Must be set before vllm is imported (§7).
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
# GPU 0 usually drives the display (~2.6 GB) -> leave it headroom; GPU 1 is headless compute.
_DEF_UTIL = "0.82" if (WORKER_MODE and WORKER_GPU == "0") else "0.90"
GPU_MEM_UTIL = float(os.environ.get("GPU_MEM_UTIL", _DEF_UTIL))
print(f"[gpu] {('WORKER gpu '+WORKER_GPU) if WORKER_MODE else ('BOTH GPUs (TP=2)' if RUN_ON_BOTH_GPUS else 'single GPU '+SINGLE_GPU_INDEX)}"
      f" -> CUDA_VISIBLE_DEVICES={VLLM_GPUS} TP_SIZE={TP_SIZE} util={GPU_MEM_UTIL}", flush=True)

# Anchor cwd to the data folder. Default is the Windows G: path (same as the HF notebook); override with
# MADS_DATA_DIR on Linux/WSL where the corpus lives elsewhere. Guarded so a missing path won't error here.
_DATA_DIR = Path(os.environ.get("MADS_DATA_DIR",
              r"G:\Other computers\My MacBook Pro\Documents\UM MADS\696 - Milestone 2\Project Data - 2 - Filtering"))
if not _DATA_DIR.exists():                       # Linux/WSL fallback: the setup script copies data to ~/mads-data
    _alt = Path.home() / "mads-data"
    if _alt.exists(): _DATA_DIR = _alt
if _DATA_DIR.exists(): os.chdir(_DATA_DIR)
HERE             = Path.cwd()
POSTS_PARQUET    = HERE / "all_posts.parquet"
COMMENTS_PARQUET = HERE / "all_comments.parquet"
# all_posts is read in §3; all_comments is ONLY needed to (re)build the assembly, which we REUSE from
# thr_comments.parquet — so it is optional here (we don't copy the 2 GB file to Linux).
assert POSTS_PARQUET.exists(), f"all_posts.parquet not found under {HERE}"

# Device for the (checkpointed) embedding anchoring stage. vLLM owns the GPU later; anchoring runs first
# and frees itself before vLLM loads. If the anchoring checkpoints already exist this is never used.
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

# --- Semantic anchoring (Stage A) ---
EMBED_MODEL        = "all-MiniLM-L6-v2"
EMBED_SIM_THRESHOLD= 0.45
EMBED_ALL_POSTS    = True

# --- LLM (Stage C) --- pinned for reproducibility
LLM_MODEL          = "Qwen/Qwen2.5-7B-Instruct-AWQ"
MODEL_TAG          = "7B-AWQ"   # REUSE the existing anchoring/assembly checkpoints in relevance_filtering-7B-AWQ
VTAG               = "-vllm"    # ALL LLM artifacts get this suffix -> isolated from the running HF (*-shard*) job
RESUME_FROM_HF     = True       # start where the HF shards left off: SKIP threads already in the HF *-shard* ledgers,
                                # and merge HF labels + vLLM labels into the final output. Set False to relabel everything.
LLM_REVISION       = None       # pin to a commit hash for a frozen run
PROMPT_VERSION     = "v3"       # v3 = terse one-line-per-id output (same prompt as the HF run)
MAX_NEW_TOKENS     = 512        # terse output is ~112 tok; 512 = safe headroom + bounds the rare runaway tail
MAX_INPUT_TOKENS   = 3072       # prompt budget (chunking keeps inputs under this)
MAX_MODEL_LEN      = int(os.environ.get("MAX_MODEL_LEN", str(MAX_INPUT_TOKENS + MAX_NEW_TOKENS)))  # vLLM ctx
# Cap concurrent sequences vLLM admits per step. vLLM's default (256) over-admits long sequences relative to
# the 4090 KV-cache budget, which triggers preemption->recompute THRASHING (throughput collapses to ~0 toks/s,
# repeated shm_broadcast 60s timeouts) on heavy thread groups. ~96 keeps the working set inside KV so big
# groups process in waves instead of thrashing. Raise toward 128-160 if light groups feel throttled.
MAX_NUM_SEQS       = int(os.environ.get("MAX_NUM_SEQS", "128"))
CHAR_BUDGET        = 5500
MAX_BODY_CHARS     = 400
CTX_FULL_LEVELS    = 3
CONTEXT_BODY_CHARS = 80

# --- Scope: which threads to send to the LLM ---
# True  = all 85K *anchored* threads (lexical OR semantic hit) — recommended with vLLM; recovers ~62% of
#         the estimated true relevant population vs ~13% for tier-1 alone.
# False = tier-1 only (~11,642 threads) — the old HF speed compromise; still valid as a fast baseline.
CLASSIFY_ANCHORED  = True

# --- Caps (None = full corpus) ---
LIMIT_POSTS        = None
MAX_THREADS        = None

# --- Checkpointing / resumability ---
CKPT_DIR           = HERE / f"relevance_filtering-{MODEL_TAG}"   # shared with HF run for UPSTREAM artifacts only
CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR            = CKPT_DIR
SAVE_EVERY_THREADS = 50    # checkpoint every 50 threads -> a crash/reboot loses <=50 threads of work.
                           # With MAX_NUM_SEQS capping concurrency, group size no longer drives GPU saturation,
                           # so 50 buys finer checkpoints at negligible throughput cost; raise for fewer flushes.
SAVE_EVERY_POSTS   = 600
BENCH_THREADS      = 500   # threads timed by the benchmark stage
CLAIM_SIZE         = SAVE_EVERY_THREADS   # DP worker (§9b): threads pulled per claim from the shared queue
RECLAIM_TIMEOUT_S  = 600   # DP worker: a dead worker's unfinished claims become re-claimable after this many seconds

print(f"[vllm-cfg] CUDA_VISIBLE_DEVICES={VLLM_GPUS} | TP_SIZE={TP_SIZE} | gpu_mem_util={GPU_MEM_UTIL} "
      f"| max_model_len={MAX_MODEL_LEN} | data={HERE}", flush=True)

[gpu] single GPU 1 -> CUDA_VISIBLE_DEVICES=1 TP_SIZE=1 util=0.9
[vllm-cfg] CUDA_VISIBLE_DEVICES=1 | TP_SIZE=1 | gpu_mem_util=0.9 | max_model_len=3584 | data=/home/freya/mads-data
CPU times: user 2.35 s, sys: 794 ms, total: 3.14 s
Wall time: 2.57 s


### Checkpoint helpers

Same crash-safe scheme as the HF notebook (immutable `part_*.parquet` + atomically-rewritten JSON ledger),
but every LLM-output name carries the `VTAG` (`-vllm`) suffix so it never collides with the HF run's files.

In [2]:
%%time
def ck(name):       return CKPT_DIR / name
def have(name):     return ck(name).exists()
def save_parquet(df, name): df.to_parquet(ck(name))
def load_parquet(name):     return pd.read_parquet(ck(name))
def save_json(obj, name):   ck(name).write_text(json.dumps(obj))
def load_json(name):        return json.loads(ck(name).read_text()) if have(name) else None
def save_json_atomic(obj, name):
    tmp = ck(name + ".tmp"); tmp.write_text(json.dumps(obj)); tmp.replace(ck(name))

def append_parts(df, subdir):
    d = ck(subdir); d.mkdir(parents=True, exist_ok=True)
    k = len(list(d.glob("part_*.parquet")))
    df.to_parquet(d / f"part_{k:05d}.parquet")
def load_parts(subdir):
    d = ck(subdir)
    if not d.exists(): return pd.DataFrame()
    files = sorted(d.glob("part_*.parquet"))
    return pd.concat((pd.read_parquet(f) for f in files), ignore_index=True) if files else pd.DataFrame()
def load_parts_all(base):    # merge a stage across ALL part-dirs whose name starts with `base`
    dfs = [load_parts(d.name) for d in sorted(CKPT_DIR.glob(f"{base}*")) if d.is_dir()]
    dfs = [x for x in dfs if len(x)]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def hf_done_threads():       # union of the HF run's per-shard comment ledgers (frozen once the HF run stops)
    s = set()
    for f in CKPT_DIR.glob("comment_done_threads-shard*.json"):
        try: s |= set(json.loads(f.read_text()) or [])
        except Exception: pass
    return s
def hf_done_posts():
    s = set()
    for f in CKPT_DIR.glob("post_done_ids-shard*.json"):
        try: s |= set(json.loads(f.read_text()) or [])
        except Exception: pass
    return s

# vLLM-namespaced LLM outputs (VTAG="-vllm") -> isolated from the HF run.
COMMENT_LABELS = f"comment_labels{VTAG}"
COMMENT_DONE   = f"comment_done_threads{VTAG}.json"
POST_LABELS    = f"post_labels{VTAG}"
POST_DONE      = f"post_done_ids{VTAG}.json"
LLM_CACHE      = f"llm_cache{VTAG}-dp{WORKER_GPU}.json" if WORKER_MODE else f"llm_cache{VTAG}.json"  # per-worker -> no clobber
REL_POSTS_OUT  = f"relevant_posts{VTAG}.parquet"
REL_COMM_OUT   = f"relevant_comments{VTAG}.parquet"
MANIFEST_OUT   = f"manifest{VTAG}.json"

# The §14 consistency-pass namespace (HF-done tier-1 threads relabeled with vLLM).
REDO_COMMENT_LABELS = "vllm_redo_comment_labels"
REDO_COMMENT_DONE   = "vllm_redo_comment_done.json"

def load_vllm_comments():
    """All-vLLM comment labels, zero HF. Globs every vLLM namespace — `comment_labels-vllm` (single/TP runs),
    `comment_labels-vllm-dp0/-dp1` (the §9b data-parallel workers) — UNION the consistency-pass relabels
    (`vllm_redo_comment_labels`). The `comment_labels-vllm*` glob deliberately EXCLUDES the HF `*-shard*` dirs."""
    return pd.concat([load_parts_all(f"comment_labels{VTAG}"), load_parts(REDO_COMMENT_LABELS)],
                     ignore_index=True).drop_duplicates("id")

print("checkpoint dir:", CKPT_DIR, "| labels:", COMMENT_LABELS, "| ledger:", COMMENT_DONE, flush=True)

checkpoint dir: /home/freya/mads-data/relevance_filtering-7B-AWQ | labels: comment_labels-vllm | ledger: comment_done_threads-vllm.json
CPU times: user 671 μs, sys: 259 μs, total: 930 μs
Wall time: 945 μs


## 2 · Detection vocabulary

Tier-1 = unambiguous (auto-trusted anchors). Candidate = deliberately broad recall net. RE2-safe so the same
pattern runs in Polars and DuckDB.

In [3]:
%%time
HIGH_PRECISION_TERMS = [
    r"congestion pricing", r"congestion toll(?:ing)?", r"congestion fee", r"congestion charge",
    r"congestion relief zone", r"\bCRZ\b", r"\bcbdtp\b",
    r"central business district toll(?:ing)?", r"traffic mobility act",
    r"hochul['’]?s? (?:congestion )?toll", r"toll(?:ing)? (?:pause|freeze)",
    r"pause(?:d|s)? the (?:congestion )?toll",
    r"\$9 (?:toll|charge|fee|congestion|cordon)", r"\$15 (?:toll|charge|fee|congestion|cordon)",
    r"(?:nine|9)[ -]?dollar (?:congestion )?toll", r"(?:fifteen|15)[ -]?dollar (?:congestion )?toll",
]
HIGH_PRECISION_PATTERN = "(?:" + "|".join(HIGH_PRECISION_TERMS) + ")"

CANDIDATE_TERMS = [
    r"\btoll", r"congestion", r"\bMTA\b", r"hochul", r"60th st", r"\$9\b", r"\$15\b",
    r"gridlock", r"traffic mobility", r"central business district", r"cordon", r"gantr",
    r"manhattan.{0,15}(?:driv|commut|toll)",
]
CANDIDATE_PATTERN = "(?:" + "|".join(CANDIDATE_TERMS) + ")"
print("Tier-1 terms:", len(HIGH_PRECISION_TERMS), "| candidate terms:", len(CANDIDATE_TERMS))

Tier-1 terms: 16 | candidate terms: 13
CPU times: user 97 μs, sys: 38 μs, total: 135 μs
Wall time: 131 μs


## 3 · Load & clean posts

In [4]:
%%time
import polars as pl
posts = pl.read_parquet(POSTS_PARQUET, columns=[
    "id","name","subreddit","title","selftext","created_utc","score","num_comments","author","permalink"])
if LIMIT_POSTS: posts = posts.head(LIMIT_POSTS)
JUNK = ["[removed]","[deleted]","[deleted by user]","[ Removed by Reddit ]"]
posts = posts.with_columns(
    pl.when(pl.col("selftext").is_in(JUNK)).then(pl.lit("")).otherwise(pl.col("selftext")).fill_null("").alias("selftext_clean")
).with_columns(
    (pl.col("title").fill_null("") + pl.lit(" ") + pl.col("selftext_clean")).alias("full_text"))
print(f"Loaded {posts.height:,} posts")

Loaded 996,115 posts
CPU times: user 782 ms, sys: 1.24 s, total: 2.03 s
Wall time: 524 ms


## 4 · Stage A — anchor detection

A *thread* is **anchored** if the post or any comment is topical, by either the lexical net or semantic
similarity to the Tier-1 centroid. These stages are CHECKPOINTED — on this machine they load instantly
from the existing `relevance_filtering-7B-AWQ/` artifacts (no GPU recompute).

In [5]:
%%time
# A1 — lexical anchors on posts (cheap; always runs)
posts = posts.with_columns([
    pl.col("full_text").str.contains("(?i)"+HIGH_PRECISION_PATTERN).alias("hp"),
    pl.col("full_text").str.contains("(?i)"+CANDIDATE_PATTERN).alias("cand"),
])
lex_anchor_posts = posts.filter(pl.col("cand"))
tier1_posts      = posts.filter(pl.col("hp"))
print(f"[A1] lexical post anchors: {lex_anchor_posts.height:,}  (Tier-1: {tier1_posts.height:,})")

[A1] lexical post anchors: 20,311  (Tier-1: 3,589)
CPU times: user 1min 10s, sys: 170 ms, total: 1min 10s
Wall time: 6.08 s


In [6]:
%%time
# A2 — semantic anchors on posts (cosine sim to Tier-1 centroid). CHECKPOINTED + chunked.
if have("sem_anchors.parquet"):
    _sa = load_parquet("sem_anchors.parquet")
    sem_anchor_post_ids = set(_sa.loc[_sa["sem"] >= EMBED_SIM_THRESHOLD, "id"])
    post_sem = dict(zip(_sa["id"], _sa["sem"].astype(float)))
    print(f"[SKIP] loaded {len(_sa):,} post sem-scores from checkpoint; anchors={len(sem_anchor_post_ids):,}", flush=True)
else:
    import torch
    from sentence_transformers import SentenceTransformer, util
    print("[A2a] loading embedding model ...", flush=True)
    emb_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    t1_texts = tier1_posts.select("full_text").to_series().to_list()
    print(f"[A2b] centroid from {len(t1_texts):,} Tier-1 posts ...", flush=True)
    t1_emb = emb_model.encode(t1_texts, batch_size=256, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False, device=DEVICE)
    centroid = t1_emb.mean(dim=0, keepdim=True); centroid = centroid / centroid.norm()
    scope = posts if EMBED_ALL_POSTS else lex_anchor_posts
    total_rows = scope.height
    print(f"[A2c] encoding {total_rows:,} posts on GPU in chunks ...", flush=True)
    DF_CHUNK_SIZE = 100_000
    ids_all, sem_all = [], []
    for offset in range(0, total_rows, DF_CHUNK_SIZE):
        pl_chunk    = scope.select(["id","full_text"]).slice(offset, DF_CHUNK_SIZE)
        chunk_texts = pl_chunk["full_text"].to_list()
        chunk_ids   = pl_chunk["id"].to_list()
        pe = emb_model.encode(chunk_texts, batch_size=256, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False, device=DEVICE)
        sem_scores = util.cos_sim(pe, centroid).squeeze(1).cpu().numpy()
        ids_all += chunk_ids; sem_all += sem_scores.astype(float).tolist()
        print(f"   [A2c] {min(offset+DF_CHUNK_SIZE, total_rows):,}/{total_rows:,}", flush=True)
    _sa = pd.DataFrame({"id": ids_all, "sem": sem_all})
    save_parquet(_sa, "sem_anchors.parquet")
    sem_anchor_post_ids = set(_sa.loc[_sa["sem"] >= EMBED_SIM_THRESHOLD, "id"])
    post_sem = dict(zip(_sa["id"], _sa["sem"].astype(float)))
    print(f"[A2] semantic post anchors (>= {EMBED_SIM_THRESHOLD}): {len(sem_anchor_post_ids):,}", flush=True)
    del emb_model, t1_emb, centroid
    try: del pe
    except NameError: pass
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("[A2] freed embedding model from GPU", flush=True)

[SKIP] loaded 996,115 post sem-scores from checkpoint; anchors=17,640
CPU times: user 682 ms, sys: 132 ms, total: 814 ms
Wall time: 810 ms


In [7]:
%%time
# A3 — lexical anchors on comments (DuckDB streams the 2 GB file). CHECKPOINTED.
if have("comment_anchors.parquet"):
    canc = load_parquet("comment_anchors.parquet")
    print(f"[SKIP] loaded {len(canc):,} comment anchors from checkpoint", flush=True)
else:
    import duckdb
    con = duckdb.connect()
    canc = con.execute(f"""
        SELECT id, link_id,
               regexp_matches(body, $hp,   'i') AS hp,
               regexp_matches(body, $cand, 'i') AS cand
        FROM read_parquet('{COMMENTS_PARQUET.as_posix()}')
        WHERE regexp_matches(body, $cand, 'i')
    """, {"hp": HIGH_PRECISION_PATTERN, "cand": CANDIDATE_PATTERN}).df()
    save_parquet(canc, "comment_anchors.parquet")
    print(f"[A3] lexical comment anchors: {len(canc):,}  (Tier-1: {int(canc['hp'].sum()):,})", flush=True)
anchor_comment_ids_by_thread = canc.groupby("link_id")["id"].apply(set).to_dict()

[SKIP] loaded 218,940 comment anchors from checkpoint
CPU times: user 1.1 s, sys: 51.7 ms, total: 1.15 s
Wall time: 1.14 s


## 5 · Capture–recapture recall estimate (no labels)

Two independent detectors at the **thread** level: lexical vs semantic. Their overlap estimates the hidden
total of topical threads (Lincoln–Petersen), hence the candidate net's recall.

In [8]:
%%time
def t3(i): return "t3_" + i
lex_threads = set(lex_anchor_posts.select("name").to_series().to_list()) | set(canc["link_id"].unique())
sem_threads = {t3(i) for i in sem_anchor_post_ids}
A, B = len(lex_threads), len(sem_threads); m = len(lex_threads & sem_threads)
union = len(lex_threads | sem_threads)
N_hat = (A * B / m) if m else None
est_recall = (union / N_hat) if N_hat else None
print(f"lexical n_A={A:,} | semantic n_B={B:,} | overlap m={m:,}")
print(f"N_hat ~ {N_hat:,.0f} | union={union:,} | est candidate-net recall ~ {est_recall:.1%}"
      if m else "overlap=0, cannot estimate")
save_json({"n_lexical": A, "n_semantic": B, "overlap": m, "union": union,
           "N_hat": (round(N_hat) if N_hat else None),
           "est_recall": (round(est_recall, 4) if est_recall else None)}, "capture_recapture.json")

if have("anchored_threads.json"):
    anchored_threads = set(load_json("anchored_threads.json"))
    print(f"[SKIP] loaded {len(anchored_threads):,} anchored threads from checkpoint")
else:
    anchored_threads = lex_threads | sem_threads
    if MAX_THREADS: anchored_threads = set(sorted(anchored_threads)[:MAX_THREADS])
    save_json(sorted(anchored_threads), "anchored_threads.json")
print(f"anchored threads to assemble: {len(anchored_threads):,}", flush=True)

if have("tier1_threads.json"):
    tier1_threads = set(load_json("tier1_threads.json"))
    print(f"[SKIP] loaded {len(tier1_threads):,} tier-1 threads from checkpoint")
else:
    t1_post_names = set(tier1_posts.select("name").to_series().to_list())
    t1_comment_threads = set(canc.loc[canc["hp"] == True, "link_id"].unique())
    tier1_threads = (t1_post_names | t1_comment_threads) & anchored_threads
    save_json(sorted(tier1_threads), "tier1_threads.json")
print(f"tier-1 threads for LLM classification: {len(tier1_threads):,}", flush=True)

# Derived: the actual set handed to the LLM (controlled by CLASSIFY_ANCHORED in §1).
# Set here (after both anchored_threads and tier1_threads exist) so all downstream cells use one variable.
classify_threads = anchored_threads if CLASSIFY_ANCHORED else tier1_threads
print(f"[scope] CLASSIFY_ANCHORED={CLASSIFY_ANCHORED} -> classify_threads={len(classify_threads):,} "
      f"({'all anchored' if CLASSIFY_ANCHORED else 'tier-1 only'})", flush=True)

lexical n_A=77,624 | semantic n_B=17,640 | overlap m=9,881
N_hat ~ 138,578 | union=85,383 | est candidate-net recall ~ 61.6%
[SKIP] loaded 85,383 anchored threads from checkpoint
anchored threads to assemble: 85,383
[SKIP] loaded 11,642 tier-1 threads from checkpoint
tier-1 threads for LLM classification: 11,642
[scope] CLASSIFY_ANCHORED=True -> classify_threads=85,383 (all anchored)
CPU times: user 57.3 ms, sys: 28.1 ms, total: 85.5 ms
Wall time: 80 ms


## 6 · Stage B — thread assembly

Pull every anchored thread's post + comments and group them. CHECKPOINTED to parquet (loads instantly here).

In [9]:
%%time
if have("thr_posts.parquet") and have("thr_comments.parquet"):
    thr_posts    = load_parquet("thr_posts.parquet")
    thr_comments = load_parquet("thr_comments.parquet")
    print(f"[SKIP] loaded assembly: {len(thr_posts):,} posts + {len(thr_comments):,} comments", flush=True)
else:
    import duckdb
    con = duckdb.connect()
    con.register("anchored", pd.DataFrame({"name": sorted(anchored_threads)}))
    thr_posts = con.execute(f"""
        SELECT p.id, p.name, p.title, p.selftext, p.subreddit, p.score, p.num_comments, p.hp_flag
        FROM (SELECT *, regexp_matches(coalesce(title,'')||' '||coalesce(selftext,''), $hp, 'i') AS hp_flag
              FROM read_parquet('{POSTS_PARQUET.as_posix()}')) p
        SEMI JOIN anchored a ON p.name = a.name
    """, {"hp": HIGH_PRECISION_PATTERN}).df()
    thr_comments = con.execute(f"""
        SELECT c.id, c.parent_id, c.link_id, c.body, c.author, c.created_utc, c.score
        FROM read_parquet('{COMMENTS_PARQUET.as_posix()}') c
        SEMI JOIN anchored a ON c.link_id = a.name
    """).df()
    save_parquet(thr_posts, "thr_posts.parquet")
    save_parquet(thr_comments, "thr_comments.parquet")
    print(f"[B] assembled {len(thr_posts):,} posts + {len(thr_comments):,} comments", flush=True)
comments_by_thread = {k: v for k, v in thr_comments.groupby("link_id")}
posts_by_name = thr_posts.set_index("name").to_dict("index")

[SKIP] loaded assembly: 84,854 posts + 5,778,767 comments
CPU times: user 9.07 s, sys: 5.07 s, total: 14.1 s
Wall time: 11.1 s


## 7 · Stage C — vLLM (lazy load)

Greedy decoding via vLLM. `ensure_llm()` builds one `LLM` engine (AWQ weights, fp16, Marlin kernels) on the
first call. vLLM does its own continuous batching, so there is **no manual batch size and no OOM-backoff** —
we just hand `llm.generate()` the whole group of prompts and it schedules them to fill the GPU. The on-disk
prompt cache (`llm_cache-vllm.json`) is flushed at every checkpoint.

In [10]:
%%time
from vllm import LLM, SamplingParams

_LLM = {"llm": None, "sp": None, "tok": None}
def ensure_llm():
    if _LLM["llm"] is None:
        print(f"[vllm] loading {LLM_MODEL} | TP={TP_SIZE} | gpu_mem_util={GPU_MEM_UTIL} "
              f"| max_model_len={MAX_MODEL_LEN} | CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}", flush=True)
        import shutil
        # enforce_eager skips torch.compile (needs the CUDA toolkit / nvcc, absent on default WSL).
        # Auto: eager when nvcc is missing; override with env VLLM_EAGER=0/1.
        _ev = os.environ.get("VLLM_EAGER")
        _eager = bool(int(_ev)) if _ev is not None else (shutil.which("nvcc") is None)
        print(f"[vllm] enforce_eager={_eager} (nvcc {'found' if shutil.which('nvcc') else 'absent'})", flush=True)
        kw = dict(model=LLM_MODEL, tokenizer=LLM_MODEL, quantization="awq_marlin", dtype="float16",
                  tensor_parallel_size=TP_SIZE, gpu_memory_utilization=GPU_MEM_UTIL,
                  max_model_len=MAX_MODEL_LEN, max_num_seqs=MAX_NUM_SEQS,   # cap concurrency -> no KV preempt thrash
                  trust_remote_code=True, seed=0, enforce_eager=_eager)
        if LLM_REVISION: kw["revision"] = LLM_REVISION
        _LLM["llm"] = LLM(**kw)
        _LLM["tok"] = _LLM["llm"].get_tokenizer()
        _LLM["sp"]  = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)
    return _LLM["llm"], _LLM["sp"], _LLM["tok"]

_cache = load_json(LLM_CACHE) or {}
def _key(prompt): return hashlib.sha1(f"{LLM_MODEL}|{PROMPT_VERSION}|{prompt}".encode()).hexdigest()
def flush_cache(): save_json_atomic(_cache, LLM_CACHE)

def _split_prompt(p, parts=2):
    """Split an over-long rendered prompt into `parts` sub-prompts that EACH repeat the instructions + post
    header and divide the comment lines between them — so every comment is still labeled (vs. truncation,
    which drops the trailing ones). Used only on the rare length-error path. Splits on the 'COMMENTS:' marker
    so the header rides along with each piece; falls back to a plain line split if the marker is absent."""
    marker = "COMMENTS:\n"
    k = p.find(marker)
    head, body = (p[:k+len(marker)], p[k+len(marker):]) if k != -1 else ("", p)
    lines = [ln for ln in body.split("\n") if ln != ""]
    if len(lines) < 2: return [p]                    # single line: indivisible (caller truncates as last resort)
    sz = math.ceil(len(lines) / parts)
    return [head + "\n".join(lines[i:i+sz]) for i in range(0, len(lines), sz)]

def llm_generate(prompts, outer=0, inner=0):
    """Cache-aware batched generation. Uncached prompts are chat-templated and handed to vLLM in ONE
    generate() call; vLLM batches them internally (continuous batching). Outputs come back in input order."""
    llm, sp, tok = ensure_llm()
    out = [None]*len(prompts); todo, idx = [], []
    for i, p in enumerate(prompts):
        k = _key(p)
        if k in _cache: out[i] = _cache[k]
        else: todo.append(p); idx.append(i)
    if todo:
        chats = [tok.apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
                 for p in todo]
        try:
            texts = [r.outputs[0].text for r in llm.generate(chats, sp)]   # COMMON PATH: no pre-tokenize
        except Exception:
            # RARE PATH ONLY. One over-long prompt makes vLLM raise VLLMValidationError and abort the WHOLE
            # batch — which previously killed the run and left the GPU idle for hours. CHAR_BUDGET bounds chunks
            # by characters, but token-dense text (CJK, emoji, code) can blow past the TOKEN budget. So only now
            # do we pay to tokenize, and instead of truncating we SPLIT each offender into pieces that fit, run
            # them, and re-join their outputs per original prompt (parse_items then sees every id). Self-healing:
            # the run continues. If nothing is actually over-long, it's a different error — re-raise, don't mask.
            _budget = MAX_INPUT_TOKENS - 128
            def _fits(s): return len(tok(s, add_special_tokens=False).get("input_ids", [])) <= _budget
            pieces, owners, nover = [], [], 0
            for t, p in enumerate(todo):
                if _fits(p):
                    pieces.append(p); owners.append(t); continue
                nover += 1
                work = _split_prompt(p, 2)
                while work:
                    s = work.pop(0)
                    if _fits(s):
                        pieces.append(s); owners.append(t); continue
                    sub = _split_prompt(s, 2)
                    if len(sub) <= 1:                # _split_prompt couldn't divide it (single comment line):
                        ids = tok(s, add_special_tokens=False).get("input_ids", [])  # indivisible -> truncate
                        pieces.append(tok.decode(ids[:_budget])); owners.append(t)   # (true last resort)
                    else:
                        work = sub + work            # divisible -> recurse on the smaller pieces
            if nover == 0:
                raise                                # not a length problem — surface the real error
            print(f"[llm] {nover} over-long prompt(s) split into fitting pieces; retrying batch", flush=True)
            pchats = [tok.apply_chat_template([{"role":"user","content":s}], tokenize=False, add_generation_prompt=True)
                      for s in pieces]
            pres = [r.outputs[0].text for r in llm.generate(pchats, sp)]
            merged = {}
            for own, txt in zip(owners, pres): merged.setdefault(own, []).append(txt)
            texts = ["\n".join(merged[t]) for t in range(len(todo))]   # re-join pieces -> one output per prompt
        for j, txt in zip(idx, texts):
            out[j] = txt; _cache[_key(prompts[j])] = txt
    return out

_STANCE_NORM = {"pro":"pro","anti":"anti","neutral":"neutral","mixed":"mixed","na":"NA"}
def _b01(s): return str(s).strip().lower() in ("1","true","yes","y","t")
def _conf(s):
    try: return max(0.0, min(1.0, float(s)))
    except Exception: return None
def parse_items(text: str) -> dict:
    """Parse the terse one-line-per-id output:
         <id> <about_topic 1/0> <stance> [<refers_to_parent 1/0>] <confidence>
    Type-aware (find stance by its label, confidence by the trailing float) so it tolerates extra
    whitespace/commas and serves both comment (5-field) and post (4-field) lines. Unparseable lines are
    skipped -> caller records parsed=False (a known failure, not a silent mislabel)."""
    res = {}
    for raw in text.splitlines():
        parts = raw.replace(",", " ").split()
        if len(parts) < 3: continue
        cid = parts[0].strip().strip("[](){}:#")
        if not cid: continue
        si = next((i for i,p in enumerate(parts[1:], start=1) if p.strip().lower() in _STANCE_NORM), None)
        if si is None or si < 2: continue
        rest = parts[si+1:]
        conf = next((c for c in (_conf(p) for p in reversed(rest)) if c is not None), 0.0)
        it = {"id": cid, "about_topic": _b01(parts[1]),
              "stance": _STANCE_NORM[parts[si].strip().lower()], "confidence": conf}
        if len(rest) >= 2: it["refers_to_parent"] = _b01(rest[0])
        res[cid] = it
    return res

CPU times: user 4.2 s, sys: 2.28 s, total: 6.48 s
Wall time: 5.93 s


In [11]:
%%time
TOPIC = ("New York City congestion pricing: the CBDTP / the ~$9 toll to drive into the Manhattan "
         "'Congestion Relief Zone' below 60th St, the MTA program, and Hochul's pause/revival of it.")

COMMENT_INSTRUCTIONS = (
 f"You label Reddit comments about {TOPIC}\n"
 "Use the thread structure to resolve what each comment refers to. A comment IS about the topic if, read in "
 "context, it engages with congestion pricing — INCLUDING short replies like 'me too', 'exactly', 'this' when "
 "they agree/disagree with a parent that is about it. A comment is NOT about the topic if it is an off-topic "
 "remark even inside a relevant thread (e.g. 'happy cake day', unrelated tangents).\n"
 "Lines shown as '(ctx, author): ...' are ancestor comments included ONLY for context — do NOT label "
 "them; only return objects for comments shown as [cID].\n"
 "For each comment decide: about_topic (1 or 0); stance in {pro, anti, neutral, mixed, NA} "
 "(pro=supports the toll, anti=opposes, neutral=on-topic but no position e.g. a question, mixed=both sides, "
 "NA=not about_topic). Resolve agreement/disagreement RELATIVE TO THE PARENT it replies to "
 "(e.g. 'me too' under an anti comment is anti). refers_to_parent=1 if you needed the parent/context to decide else 0. "
 "confidence in 0..1. If a referent is '[deleted]'/'[removed]' and unresolvable, set about_topic=0, "
 "stance=NA, low confidence.\n"
 "OUTPUT FORMAT — output ONE line per [cID] and NOTHING ELSE (no JSON, no header, no prose). "
 "Each line is exactly five space-separated fields:\n"
 "<cID> <about_topic 1 or 0> <stance> <refers_to_parent 1 or 0> <confidence>\n"
 "Example:\nk8j3l2m 1 pro 0 0.85\nk9a1b2c 0 NA 0 0.40\n"
 "Output one line for EVERY [cID] shown, using the id EXACTLY as printed inside the [ ] (no brackets).")

POST_INSTRUCTIONS = (
 f"You label whether each Reddit POST is about {TOPIC}\n"
 "For each post decide: about_topic (1 or 0), stance in {pro, anti, neutral, mixed, NA}, confidence in 0..1.\n"
 "OUTPUT FORMAT — output ONE line per [pID] and NOTHING ELSE (no JSON, no prose). "
 "Each line is exactly four space-separated fields:\n"
 "<pID> <about_topic 1 or 0> <stance> <confidence>\n"
 "Example:\nt3_abc123 1 anti 0.90\n"
 "Output one line for EVERY [pID] shown, using the id EXACTLY as printed inside the [ ] (no brackets).")
print("prompts ready, version", PROMPT_VERSION)

prompts ready, version v3
CPU times: user 75 μs, sys: 0 ns, total: 75 μs
Wall time: 68.9 μs


## 8 · Thread rendering (lineage-bounded chunks)

Identical chunking to the HF notebook: each comment is labeled in its **lineage context** (root→parent shown
as read-only `(ctx, …)` lines), chunks bounded per comment by `CHAR_BUDGET`, ancestor chain re-seeded across
chunk splits so no comment loses its parent.

In [12]:
%%time
def _clean(b):
    b = (b or "")
    return b if b not in JUNK else "[removed]"

def build_forest(cdf):
    nodes = {r.id: {"id": r.id, "parent_id": r.parent_id, "author": r.author,
                    "body": _clean(r.body), "created_utc": r.created_utc, "children": []}
             for r in cdf.itertuples()}
    roots = []
    for n in nodes.values():
        pid = n["parent_id"]
        if isinstance(pid, str) and pid.startswith("t1_") and pid[3:] in nodes:
            nodes[pid[3:]]["children"].append(n)
        else:
            roots.append(n)
    def srt(ns):
        ns.sort(key=lambda x: x["created_utc"])
        for x in ns: srt(x["children"])
    srt(roots)
    return nodes, roots

def _target_ids(nodes, roots, post_is_anchor, anchor_cids):
    if post_is_anchor or not anchor_cids:
        return set(nodes)
    keep = set()
    for a in anchor_cids:
        if a not in nodes: continue
        cur = a
        while cur in nodes:
            keep.add(cur); p = nodes[cur]["parent_id"]
            cur = p[3:] if isinstance(p, str) and p.startswith("t1_") and p[3:] in nodes else None
        stack = [nodes[a]]
        while stack:
            x = stack.pop(); keep.add(x["id"]); stack += x["children"]
    return keep

def _parent_map(nodes):
    pm = {}
    for n in nodes.values():
        pid = n["parent_id"]
        pm[n["id"]] = pid[3:] if isinstance(pid, str) and pid.startswith("t1_") and pid[3:] in nodes else None
    return pm

def _depths(nodes, pm):
    d = {}
    def dep(i):
        if i in d: return d[i]
        p = pm[i]; d[i] = 0 if p is None else dep(p)+1; return d[i]
    for i in nodes: dep(i)
    return d

def render_chunks(post, cdf, anchor_cids):
    nodes, roots = build_forest(cdf)
    target = _target_ids(nodes, roots, post.get("post_is_anchor", False), anchor_cids)
    pm = _parent_map(nodes); depth = _depths(nodes, pm)
    header = f"POST [{post['id']}] {(post.get('title') or '')[:200]}\n{_clean(post.get('selftext'))[:800]}\n\nCOMMENTS:\n"

    def ind(i): return "  " * min(depth[i], 10)
    def ancestors(i):
        ch = []; cur = pm[i]
        while cur is not None: ch.append(cur); cur = pm[cur]
        return ch[::-1]
    def tgt_line(i):
        n = nodes[i]; return f"{ind(i)}[{i}] ({n['author']}): {n['body'][:MAX_BODY_CHARS]}"
    def ctx_line(i, dist):
        n = nodes[i]; cap = MAX_BODY_CHARS if dist <= CTX_FULL_LEVELS else CONTEXT_BODY_CHARS
        return f"{ind(i)}(ctx, {n['author']}): {n['body'][:cap]}"

    order = []
    for r in roots:
        st = [r]
        while st:
            x = st.pop()
            if x["id"] in target: order.append(x["id"])
            st.extend(reversed(x["children"]))
    if not order: return []

    hdr_len = len(header)
    chunks, lines, ids, shown, size = [], [], [], set(), hdr_len
    def flush():
        if ids: chunks.append((header + "\n".join(lines), list(ids)))
    for t in order:
        chain = ancestors(t)
        addl = [(a, ctx_line(a, depth[t]-depth[a])) for a in chain if a not in shown]
        tl   = tgt_line(t)
        cost = sum(len(l)+1 for _, l in addl) + len(tl) + 1
        if size + cost > CHAR_BUDGET and ids:
            flush(); lines, ids, shown, size = [], [], set(), hdr_len
            addl = [(a, ctx_line(a, depth[t]-depth[a])) for a in chain]
        for a, l in addl:
            lines.append(l); shown.add(a); size += len(l)+1
        lines.append(tl); shown.add(t); ids.append(t); size += len(tl)+1
    flush()
    return chunks

def thread_to_chunks(name):
    post = posts_by_name.get(name, {"id": name.replace("t3_",""), "title": "", "selftext": "", "hp_flag": False})
    post = dict(post); post["post_is_anchor"] = bool(post.get("hp_flag")) or (name in lex_threads) or (name in sem_threads)
    cdf = comments_by_thread.get(name)
    if cdf is None or len(cdf) == 0: return [], []
    anchor_cids = anchor_comment_ids_by_thread.get(name, set())
    chunks, idlists = [], []
    for text, ids in render_chunks(post, cdf, anchor_cids):
        chunks.append(COMMENT_INSTRUCTIONS + "\n\n" + text); idlists.append(ids)
    return chunks, idlists
print("rendering ready")

rendering ready
CPU times: user 91 μs, sys: 0 ns, total: 91 μs
Wall time: 86.3 μs


## 8b · Resumable classification functions

Same resumable ledger/part-file scheme as the HF notebook, but each checkpoint group's chunks are handed to
vLLM in **one** `llm_generate()` call (vLLM batches internally). Both functions resume from disk.

In [13]:
%%time
def classify_comment_threads(thread_names, save_every=SAVE_EVERY_THREADS, bench_limit=None,
                             labels_name=None, done_name=None, remaining_fn=None, cum=None):
    labels_name = labels_name or COMMENT_LABELS     # override to write to a separate namespace (the redo pass)
    done_name   = done_name   or COMMENT_DONE
    done = set(load_json(done_name) or [])
    todo = [n for n in thread_names if n not in done]
    if bench_limit is not None: todo = todo[:bench_limit]
    print(f"comment threads: {len(done):,} done | {len(todo):,} to process now", flush=True)
    if not todo: return
    ensure_llm()
    t0 = time.time(); start_done = len(done); prev_el = 0.0
    # `cum` (DP worker) is a caller-owned dict that persists ACROSS the many per-claim calls, so the cumulative
    # rate spans the whole worker session instead of resetting each claim. Single-engine §9 passes cum=None.
    if cum is not None:
        cum.setdefault("t0", t0); cum.setdefault("done0", start_done)
    for gs in range(0, len(todo), save_every):
        group = todo[gs:gs+save_every]
        chunks, idlists = [], []
        for name in group:
            c, il = thread_to_chunks(name)
            chunks += c; idlists += il
        outs = llm_generate(chunks, outer=gs, inner=0) if chunks else []
        rows = []
        for out, ids in zip(outs, idlists):
            parsed = parse_items(out)
            for cid in ids:
                it = parsed.get(cid, {})
                rows.append({"id": cid,
                    "about_topic": bool(it.get("about_topic", False)),
                    "stance": it.get("stance", "NA"),
                    "refers_to_parent": bool(it.get("refers_to_parent", False)),
                    "confidence": float(it.get("confidence", 0.0) or 0.0),
                    "parsed": bool(it)})
        if rows: append_parts(pd.DataFrame(rows), labels_name)
        done |= set(group); save_json_atomic(sorted(done), done_name); flush_cache()
        now = time.time(); el = now - t0
        batch_rate = (el - prev_el) / max(len(group), 1); prev_el = el   # this group's own threads/sec
        if cum is not None:                                             # session-cumulative (across all claims)
            cum_el = now - cum["t0"]; cum_n = len(done) - cum["done0"]
        else:                                                           # single-call cumulative (§9 path)
            cum_el = el; cum_n = len(done) - start_done
        cum_rate = cum_el / max(cum_n, 1)
        # `remaining_fn` (DP worker) reports the TRUE shared-queue remaining across BOTH workers, recomputed from
        # the just-written ledgers; without it (single-engine §9) fall back to this run's batch-local count.
        _left = (f"{remaining_fn():,} left (shared queue)" if remaining_fn is not None
                 else f"{len(todo)-(gs+len(group)):,} left this run")
        print(f"[ckpt] +{len(group)} -> {len(done):,} threads | {cum_el:.0f}s | {cum_rate:.2f}s/thread cum "
              f"| {batch_rate:.2f}s/thread batch | {_left}", flush=True)

def classify_posts(ap_df, save_every=SAVE_EVERY_POSTS):
    done = set(load_json(POST_DONE) or [])
    todo = ap_df[~ap_df["id"].isin(done)].reset_index(drop=True)
    print(f"posts: {len(done):,} done | {len(todo):,} to process now", flush=True)
    if len(todo) == 0: return
    ensure_llm()
    t0 = time.time()
    for gs in range(0, len(todo), save_every):
        grp = todo.iloc[gs:gs+save_every]
        prompts, batches = [], []
        for s in range(0, len(grp), 12):
            chunk = grp.iloc[s:s+12]
            body = "\n".join(f"[{r.id}] {str(r.title)[:200]} :: {str(r.selftext_clean)[:500]}" for r in chunk.itertuples())
            prompts.append(POST_INSTRUCTIONS + "\n\nPOSTS:\n" + body); batches.append(list(chunk["id"]))
        lab = {}
        for out in llm_generate(prompts, outer=gs, inner=0): lab.update(parse_items(out))
        rows = []
        for r in grp.itertuples():
            it = lab.get(r.id, {})
            rows.append({"id": r.id,
                "about_topic": bool(it.get("about_topic", bool(r.hp))),
                "stance": it.get("stance", "NA"),
                "confidence": float(it.get("confidence", 0.0) or 0.0)})
        append_parts(pd.DataFrame(rows), POST_LABELS)
        done |= set(grp["id"]); save_json_atomic(sorted(done), POST_DONE); flush_cache()
        print(f"[ckpt] posts {len(done):,} done | {time.time()-t0:.0f}s", flush=True)
print("classification functions ready")

classification functions ready
CPU times: user 66 μs, sys: 7 μs, total: 73 μs
Wall time: 68.7 μs


## 9b · Dynamic data-parallel worker (only runs when `WORKER_MODE`)

**Normal mode (`WORKER_MODE = False`): this cell does nothing** — execution falls through to the benchmark and the
single-engine §9 below.

**Worker mode (`WORKER_MODE = True`): this cell IS the run.** It pulls `CLAIM_SIZE` threads at a time from a shared
claim file (`dp_claims.json`, guarded by an `fcntl` lock), labels them into this worker's own namespace
(`comment_labels-vllm-dp{WORKER_GPU}`), releases them, and repeats until the queue is empty — then stops (so the
benchmark/§9/§11/§14 below are skipped in worker copies; do the merge from the **main** notebook).

Run two copies (GPU 0 and GPU 1); they share the one queue, so neither sits idle when the other finishes early.
**Interrupt** to pause a worker — it releases its in-flight claim and the engine stays loaded, so re-running this
cell resumes instantly. A worker killed outright has its unfinished claim reclaimed after `RECLAIM_TIMEOUT_S`.

In [14]:
%%time
# Claim-queue helpers (defined always; the loop at the bottom runs only in WORKER_MODE).
import contextlib
DP_LABELS = f"comment_labels{VTAG}-dp{WORKER_GPU}"          # this worker's private label part-dir
DP_DONE   = f"comment_done_threads{VTAG}-dp{WORKER_GPU}.json"
DP_CLAIMS = "dp_claims.json"                                # shared across workers
WID       = f"gpu{WORKER_GPU}"

@contextlib.contextmanager
def _claim_lock():
    # fcntl.flock is Linux-only and auto-releases if the process dies. Held ONLY for the ms-long claim/release
    # bookkeeping, never during generation. On Windows (smoke) fcntl is absent -> no-op (single-process is safe).
    try:
        import fcntl
        f = open(ck("dp_claims.lock"), "w")
        try: fcntl.flock(f, fcntl.LOCK_EX); yield
        finally: fcntl.flock(f, fcntl.LOCK_UN); f.close()
    except ImportError:
        yield

def _dp_all_done():
    # union of every vLLM done-ledger (this worker's, the other worker's, and any single/TP -vllm run) + HF shards
    s = hf_done_threads()
    for fp in CKPT_DIR.glob(f"comment_done_threads{VTAG}*.json"):
        try: s |= set(json.loads(fp.read_text()) or [])
        except Exception: pass
    return s

_DP_PENDING = None
def _dp_pending():
    # classify_threads and the HF-done set are FROZEN for the run, so compute the pending order ONCE and cache it.
    # (Calling hf_done_threads() — which reads ledger files — per element here cost ~85k disk reads PER claim.)
    global _DP_PENDING
    if _DP_PENDING is None:
        _hd = hf_done_threads()
        _DP_PENDING = [t for t in sorted(classify_threads) if t not in _hd]
    return _DP_PENDING

def _claim_next(k):
    with _claim_lock():
        claims = load_json(DP_CLAIMS) or {}
        now = time.time(); done = _dp_all_done()
        active = {t for t, (w, ts) in claims.items() if now - ts < RECLAIM_TIMEOUT_S}
        batch = []
        for t in _dp_pending():
            if t in done or t in active: continue
            batch.append(t)
            if len(batch) >= k: break
        for t in batch: claims[t] = [WID, now]
        if batch: save_json_atomic(claims, DP_CLAIMS)
        return batch

def _release(batch):
    if not batch: return
    with _claim_lock():
        claims = load_json(DP_CLAIMS) or {}
        for t in batch: claims.pop(t, None)
        save_json_atomic(claims, DP_CLAIMS)

if WORKER_MODE:
    _PENDING_SET = set(_dp_pending())                       # fixed for the run (classify_threads + HF-done are frozen)
    def _queue_remaining(): return len(_PENDING_SET - _dp_all_done())   # TRUE shared-queue remaining (counts BOTH workers)
    print(f"[dp-worker {WID}] {_queue_remaining():,} threads remaining in shared queue; claiming {CLAIM_SIZE} at a time", flush=True)
    ensure_llm()
    _n = 0; batch = []; _cum = {}        # _cum persists across claims -> cum rate spans the whole worker session
    try:
        while True:
            batch = _claim_next(CLAIM_SIZE)
            if not batch:
                print(f"[dp-worker {WID}] queue empty — finished. labeled {_n:,} threads this session.", flush=True); break
            try:
                classify_comment_threads(batch, save_every=CLAIM_SIZE, labels_name=DP_LABELS, done_name=DP_DONE,
                                         remaining_fn=_queue_remaining, cum=_cum)
                _n += len(batch)
            finally:
                _release(batch)            # release whether the batch finished or was interrupted mid-way
    except KeyboardInterrupt:
        _release(batch)
        print(f"[dp-worker {WID}] interrupted; released {len(batch)} claimed thread(s). Engine stays loaded — "
              f"re-run THIS cell to resume.", flush=True)
    else:
        print(f"[dp-worker {WID}] done. Merge from the MAIN notebook (WORKER_MODE=False).", flush=True)
        raise SystemExit(f"[dp-worker {WID}] worker finished — later cells are skipped in worker mode (expected).")
else:
    print("[dp-worker] WORKER_MODE=False -> skipping (this notebook runs the single-engine §8c/§9 path below)", flush=True)

[dp-worker] WORKER_MODE=False -> skipping (this notebook runs the single-engine §8c/§9 path below)
CPU times: user 0 ns, sys: 1.03 ms, total: 1.03 ms
Wall time: 835 μs


## 8c · Benchmark / timing stage

Times **(a)** the vLLM engine load and **(b)** classifying the next `BENCH_THREADS` unprocessed threads, then
extrapolates a full-run ETA. Benchmarked threads are checkpointed like any others (no wasted work).

In [15]:
%%time
_skip = hf_done_threads() if RESUME_FROM_HF else set()          # threads the HF shards already labeled
order = [n for n in sorted(classify_threads) if n not in _skip]  # vLLM only does what HF hasn't
done_now = _dp_all_done()       # HF + ALL vLLM ledgers (incl. the §9b DP workers' -dp0/-dp1) -> skip finished threads
remaining = [n for n in order if n not in done_now]
n_ran = min(BENCH_THREADS, len(remaining))
print(f"[vllm] {len(set(order) & done_now):,} of {len(order):,} already labeled (HF + vLLM incl. DP workers) | "
      f"{len(remaining):,} remaining -> benchmark runs next {n_ran} (BENCH_THREADS={BENCH_THREADS})", flush=True)

_t = time.time(); ensure_llm(); load_s = time.time() - _t
print(f"[bench] vLLM engine load: {load_s:.1f}s", flush=True)

_t = time.time()
classify_comment_threads(order, save_every=SAVE_EVERY_THREADS, bench_limit=BENCH_THREADS)
run_s = time.time() - _t
if n_ran:
    per = run_s / n_ran
    print(f"[bench] {n_ran} threads in {run_s:.0f}s = {per:.2f}s/thread", flush=True)
    print(f"[bench] EXTRAPOLATED {len(remaining):,} threads remaining ~ {per*len(remaining)/3600:.1f} h "
          f"(one-time; cache + checkpoints make reruns free)", flush=True)
else:
    print("[bench] nothing left to benchmark — all threads already processed", flush=True)

[vllm] 81,623 of 81,623 already labeled (HF + vLLM incl. DP workers) | 0 remaining -> benchmark runs next 0 (BENCH_THREADS=500)
[vllm] loading Qwen/Qwen2.5-7B-Instruct-AWQ | TP=1 | gpu_mem_util=0.9 | max_model_len=3584 | CUDA_VISIBLE_DEVICES=1
[vllm] enforce_eager=False (nvcc found)
INFO 06-11 18:30:51 [utils.py:278] non-default args: {'tokenizer': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 3584, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 128, 'disable_log_stats': True, 'quantization': 'awq_marlin', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}


INFO 06-11 18:30:51 [model.py:617] Resolved architecture: Qwen2ForCausalLM
INFO 06-11 18:30:51 [model.py:1752] Using max model len 3584
INFO 06-11 18:30:53 [awq_marlin.py:268] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-11 18:30:53 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files: 100%|██████████| 2/2 [00:00<00:00, 15.57it/s]

INFO 06-11 18:30:53 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-11 18:30:53 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


WARNING 06-11 18:30:55 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: WSL is detected and NVML is not compatible with fork
(EngineCore pid=279575) INFO 06-11 18:31:01 [core.py:112] Initializing a V1 LLM engine (v0.22.1) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=3584, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=awq_marlin, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache

...

[ckpt] +50 -> 22,732 threads | 38s | 0.10s/thread cum | 0.72s/thread batch | 100 left this run


Processed prompts: 100%|██████████| 135/135 [00:23<00:00,  5.65it/s, est. speed input: 9434.41 toks/s, output: 1724.03 toks/s]


[ckpt] +50 -> 22,782 threads | 63s | 0.14s/thread cum | 0.49s/thread batch | 50 left this run


Processed prompts: 100%|██████████| 191/191 [00:40<00:00,  4.73it/s, est. speed input: 8406.56 toks/s, output: 1653.49 toks/s]


[ckpt] +50 -> 22,832 threads | 104s | 0.21s/thread cum | 0.82s/thread batch | 0 left this run
[bench] nothing left to benchmark — all threads already processed
CPU times: user 7.21 s, sys: 1.61 s, total: 8.82 s
Wall time: 2min 20s


## 9 · Classify comments (per-comment, context-resolved, resumable)

Processes every remaining tier-1 thread, checkpointing every `SAVE_EVERY_THREADS`. Interrupt and re-run to resume.

In [16]:
%%time
# Skip everything already labeled by HF AND by any vLLM run — including the §9b data-parallel workers
# (comment_labels-vllm-dp0/-dp1). So after the workers drain the queue, this stage is a clean no-op.
_done = _dp_all_done() if RESUME_FROM_HF else (_dp_all_done() - hf_done_threads())
pending = [t for t in sorted(classify_threads) if t not in _done]
print(f"[C-comments] {len(_done & set(classify_threads)):,} already labeled (HF + vLLM incl. DP workers); "
      f"vLLM will process {len(pending):,}", flush=True)
classify_comment_threads(pending, save_every=SAVE_EVERY_THREADS)
comments_df = load_vllm_comments()   # all-vLLM: comment_labels-vllm (fresh) + vllm_redo (HF-done relabels); no HF
print(f"[C-comments] {len(comments_df):,} labeled (vLLM only) | about_topic={int(comments_df.about_topic.sum()):,} "
      f"| parse_fail={int((~comments_df.parsed).sum()):,}", flush=True)

[C-comments] 85,383 already labeled (HF + vLLM incl. DP workers); vLLM will process 0
comment threads: 22,832 done | 0 to process now
[C-comments] 5,778,767 labeled (vLLM only) | about_topic=382,562 | parse_fail=1,036,120
CPU times: user 2.51 s, sys: 1.34 s, total: 3.84 s
Wall time: 3.6 s


## 10 · Classify posts (relevance + stance, resumable)

In [17]:
%%time
anchor_post_names = (set(lex_anchor_posts.select('name').to_series().to_list())
                     | {t3(i) for i in sem_anchor_post_ids}) & classify_threads
ap_all = posts.filter(pl.col("name").is_in(list(anchor_post_names))).select(
        ["id","name","title","selftext_clean","subreddit","hp"]).to_pandas()
_skip_p = hf_done_posts() if RESUME_FROM_HF else set()
classify_posts(ap_all[~ap_all["id"].isin(_skip_p)].reset_index(drop=True), save_every=SAVE_EVERY_POSTS)

_lab = load_parts_all("post_labels").drop_duplicates("id")   # HF + vLLM merged
ap = ap_all.merge(_lab, on="id", how="left")
ap["about_topic"] = ap["about_topic"].fillna(ap["hp"]).astype(bool)
ap["stance"]      = ap["stance"].fillna("NA")
ap["confidence"]  = ap["confidence"].fillna(0.0)
print(f"[C-posts] {len(ap):,} posts labeled (HF+vLLM) | about_topic={int(ap.about_topic.sum()):,}", flush=True)

posts: 5,566 done | 25,782 to process now


Processed prompts: 100%|██████████| 50/50 [00:07<00:00,  6.58it/s, est. speed input: 8572.37 toks/s, output: 1244.26 toks/s]

[ckpt] posts 6,166 done | 8s



Processed prompts: 100%|██████████| 50/50 [00:08<00:00,  6.10it/s, est. speed input: 9014.53 toks/s, output: 1135.68 toks/s]


[ckpt] posts 6,766 done | 16s


...

[ckpt] posts 30,166 done | 292s


Processed prompts: 100%|██████████| 50/50 [00:06<00:00,  7.15it/s, est. speed input: 8781.27 toks/s, output: 1303.44 toks/s]


[ckpt] posts 30,766 done | 299s


Processed prompts: 100%|██████████| 49/49 [00:07<00:00,  6.67it/s, est. speed input: 9083.29 toks/s, output: 1187.46 toks/s]


[ckpt] posts 31,348 done | 307s
[C-posts] 31,348 posts labeled (HF+vLLM) | about_topic=3,363
CPU times: user 12.8 s, sys: 4.6 s, total: 17.4 s
Wall time: 5min 7s


## 11 · Assemble outputs + manifest

Single-process assembly (no shard merge). All outputs carry the `-vllm` suffix so they sit alongside, and never
overwrite, the HF run's `relevant_*.parquet` / `manifest.json`.

In [18]:
%%time
comments_df = load_vllm_comments()   # all-vLLM comments (fresh + redo), no HF labels
ap = (posts.filter(pl.col("name").is_in(list(anchor_post_names)))
          .select(["id","name","title","selftext_clean","subreddit","hp"]).to_pandas()
          .merge(load_parts(POST_LABELS).drop_duplicates("id"), on="id", how="left"))
ap["about_topic"] = ap["about_topic"].fillna(ap["hp"]).astype(bool)
ap["stance"]      = ap["stance"].fillna("NA")
ap["confidence"]  = ap["confidence"].fillna(0.0)

rel_posts = ap[ap["about_topic"]].copy(); rel_posts["matched_tier1"] = rel_posts["hp"]
save_parquet(rel_posts, REL_POSTS_OUT)

rel_comments = comments_df[comments_df["about_topic"]].merge(
    thr_comments[["id","link_id","author","score","created_utc"]], on="id", how="left")
save_parquet(rel_comments, REL_COMM_OUT)

manifest = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"), "engine": "vllm", "device": f"CUDA_VISIBLE_DEVICES={VLLM_GPUS}",
    "llm_model": LLM_MODEL, "model_tag": MODEL_TAG, "vtag": VTAG, "llm_revision": LLM_REVISION,
    "prompt_version": PROMPT_VERSION, "tensor_parallel_size": TP_SIZE, "max_model_len": MAX_MODEL_LEN,
    "embed_model": EMBED_MODEL, "embed_sim_threshold": EMBED_SIM_THRESHOLD,
    "anchored_threads": len(anchored_threads),
    "capture_recapture": load_json("capture_recapture.json") or {},
    "n_relevant_posts": int(len(rel_posts)), "n_relevant_comments": int(len(rel_comments)),
    "stance_posts": rel_posts["stance"].value_counts().to_dict(),
    "stance_comments": rel_comments["stance"].value_counts().to_dict(),
}
save_json(manifest, MANIFEST_OUT)
print(json.dumps(manifest, indent=2))

{
  "generated_at": "2026-06-11 18:38:28",
  "engine": "vllm",
  "device": "CUDA_VISIBLE_DEVICES=1",
  "llm_model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
  "model_tag": "7B-AWQ",
  "vtag": "-vllm",
  "llm_revision": null,
  "prompt_version": "v3",
  "tensor_parallel_size": 1,
  "max_model_len": 3584,
  "embed_model": "all-MiniLM-L6-v2",
  "embed_sim_threshold": 0.45,
  "anchored_threads": 85383,
  "capture_recapture": {
    "n_lexical": 77624,
    "n_semantic": 17640,
    "overlap": 9881,
    "union": 85383,
    "N_hat": 138578,
    "est_recall": 0.6161
  },
  "n_relevant_posts": 3363,
  "n_relevant_comments": 382562,
  "stance_posts": {
    "anti": 1093,
    "pro": 945,
    "neutral": 729,
    "NA": 509,
    "mixed": 87
  },
  "stance_comments": {
    "pro": 155863,
    "anti": 145922,
    "neutral": 76022,
    "mixed": 4564,
    "NA": 191
  }
}
CPU times: user 3.9 s, sys: 1.93 s, total: 5.84 s
Wall time: 5.07 s


## 12 · Label-free validation (pseudo-labels)

Tier-1 posts should be ~all `about_topic=True`; a keyword-free deep-tail sample should be ~all `False`.

In [19]:
%%time
t1_ids = set(tier1_posts.select("id").to_series().to_list()) & set(ap["id"])
if t1_ids:
    sub = ap[ap["id"].isin(t1_ids)]
    print(f"Tier-1 posts judged about_topic: {sub.about_topic.mean():.1%} of {len(sub):,} (want ~100%)")

tail = posts.filter(~pl.col("cand")).select(["id","title","selftext_clean"]).sample(
        n=min(60, posts.filter(~pl.col('cand')).height), seed=0).to_pandas()
prompts = [POST_INSTRUCTIONS + "\n\nPOSTS:\n" + f"[{r.id}] {str(r.title)[:200]} :: {str(r.selftext_clean)[:500]}"
           for r in tail.itertuples()]
lab = {}
for out in llm_generate(prompts): lab.update(parse_items(out))
flush_cache()
fp = np.mean([bool(lab.get(i,{}).get("about_topic", False)) for i in tail["id"]]) if len(tail) else float("nan")
print(f"Deep-tail posts judged about_topic: {fp:.1%} of {len(tail):,} (want ~0%)")

Tier-1 posts judged about_topic: 80.2% of 3,588 (want ~100%)
Deep-tail posts judged about_topic: 0.0% of 60 (want ~0%)
CPU times: user 253 ms, sys: 280 ms, total: 533 ms
Wall time: 205 ms


## 13 · Summary

In [20]:
%%time
print("=== RELEVANT POSTS (vLLM) ===", len(rel_posts))
print(rel_posts["stance"].value_counts().to_string())
print("\n=== RELEVANT COMMENTS (vLLM) ===", len(rel_comments))
print(rel_comments["stance"].value_counts().to_string())
print("\nrefers_to_parent (context-resolved):",
      f"{rel_comments['refers_to_parent'].mean():.1%}" if len(rel_comments) else "n/a")
print("\nSample relevant comments:")
for r in rel_comments.sample(min(10, len(rel_comments)), random_state=0).itertuples():
    print(f"  [{r.stance:7}] ref={r.refers_to_parent}  {str(thr_comments.set_index('id').loc[r.id,'body'])[:90]}")

=== RELEVANT POSTS (vLLM) === 3363
stance
anti       1093
pro         945
neutral     729
NA          509
mixed        87

=== RELEVANT COMMENTS (vLLM) === 382562
stance
pro        155863
anti       145922
neutral     76022
mixed        4564
NA            191

refers_to_parent (context-resolved): 37.5%

Sample relevant comments:
  [pro    ] ref=True  Residential parking would be amazing
  [anti   ] ref=True  It’s little consolation for the person who died that they were killed due to an “accident”
  [pro    ] ref=False  I have gone to the city plenty of times for non-work reasons and I’ve never had a need to 
  [pro    ] ref=True  Lmao this is why American social services are so underfunded. In Europe, they acknowledge 
  [anti   ] ref=False  Just another tax for the middle class, this won’t deter people from driving into Manhattan
  [anti   ] ref=False  It’s not just about pollution, it’s about reducing the number of cars in the city and prop
  [neutral] ref=False  Are the pace of the

In [ ]:
labeled_comments = load_vllm_comments().merge(
    thr_comments[["id","link_id","author","score","created_utc"]], on="id", how="left")
labeled_comments["thread_tier1"] = labeled_comments["link_id"].isin(tier1_threads)
save_parquet(labeled_comments, f"labeled_comments{VTAG}.parquet")
labeled_posts = ap.copy()
labeled_posts["matched_tier1"] = labeled_posts["hp"]
labeled_posts["thread_tier1"]  = labeled_posts["name"].isin(tier1_threads)
save_parquet(labeled_posts, f"labeled_posts{VTAG}.parquet")

gate = labeled_comments["thread_tier1"] | labeled_comments["about_topic"]
print(f"comments total: {len(labeled_comments):,} | relevant: {int(labeled_comments.about_topic.sum()):,} "
      f"| Tier-1-thread ∪ relevant gate: {int(gate.sum()):,}")
print(f"Tier-1 threads: {len(tier1_threads):,} | Tier-1 POSTS (hp): {tier1_posts.height:,} "
      f"| posts labeled: {len(ap):,} | posts relevant: {int(ap.about_topic.sum()):,}")

## 14 · Consistency pass — relabel the HF-done threads with vLLM (ON by default)

The main run (§9) labeled only the threads the HF shards hadn't reached, so the §11 outputs are a **mix** of
HF-labeled and vLLM-labeled comments. This pass re-labels the HF-completed threads **with vLLM** so the whole
dataset is single-engine. It writes to a **separate namespace** (`vllm_redo_*`) and produces a separate
**all-vLLM consistent** output — it does **not** overwrite the mixed §11 output. Long-running but fully
resumable: watch the rate and interrupt any time (you keep the mixed output + whatever redo finished). Set
`REDO_HF_WITH_VLLM = False` below to skip it.

In [22]:
%%time
REDO_HF_WITH_VLLM = True     # ON by default -> one consistent all-vLLM dataset. Set False to keep only the §11 mix.
if REDO_HF_WITH_VLLM and RESUME_FROM_HF and not WORKER_MODE:   # worker copies never run the merge/redo
    redo_threads = sorted(hf_done_threads() & set(classify_threads))
    print(f"[redo] relabeling {len(redo_threads):,} HF-done threads with vLLM (separate vllm_redo_* namespace) ...", flush=True)
    classify_comment_threads(redo_threads, save_every=SAVE_EVERY_THREADS,
                             labels_name=REDO_COMMENT_LABELS, done_name=REDO_COMMENT_DONE)
    # All-vLLM consistent comments = threads vLLM did fresh (comment_labels-vllm) + the redone HF threads.
    cons = load_vllm_comments()
    rel_c = cons[cons["about_topic"]].merge(
        thr_comments[["id","link_id","author","score","created_utc"]], on="id", how="left")
    save_parquet(rel_c, "relevant_comments-vllm-consistent.parquet")
    save_json({"generated_at": time.strftime("%Y-%m-%d %H:%M:%S"), "engine": "vllm-consistent",
               "llm_model": LLM_MODEL, "prompt_version": PROMPT_VERSION,
               "n_comments_labeled": int(len(cons)), "n_relevant_comments": int(len(rel_c)),
               "stance_comments": rel_c["stance"].value_counts().to_dict()}, "manifest-vllm-consistent.json")
    print(f"[redo] all-vLLM consistent: {len(cons):,} comments labeled | relevant={len(rel_c):,} "
          f"-> relevant_comments-vllm-consistent.parquet", flush=True)
else:
    print("[redo] skipped (needs REDO_HF_WITH_VLLM and RESUME_FROM_HF both True)", flush=True)

[redo] relabeling 3,760 HF-done threads with vLLM (separate vllm_redo_* namespace) ...
comment threads: 3,760 done | 0 to process now
[redo] all-vLLM consistent: 5,778,767 comments labeled | relevant=382,562 -> relevant_comments-vllm-consistent.parquet
CPU times: user 3.81 s, sys: 1.34 s, total: 5.15 s
Wall time: 4.65 s


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>